In [ ]:
# ..... paralog list across species ..... #
# primate, mouse, tree-shrew comparisons

In [2]:
library(data.table)
library(dplyr)

In [3]:
# get list of paralogs 
tab1 = read.delim('/vault/suresh/consensus_interneuron_taxonomy/homologs/human_paralogs.txt', sep = '\t')
tab1 <- tab1[(tab1[,1]!='' & tab1[,2]!='' & tab1[,4]!='' & tab1[,4]!='gene_split'),]
tab1 <- tab1[!duplicated(tab1),]
colnames(tab1)[1:2] = c('sp1_gene', 'sp2_gene')

# human - NHP
paralog_df1 = paste0(unlist(tab1$sp1_gene), '_', unlist(tab1$sp2_gene))

# NHP - human
paralog_df2 = paralog_df1

head(paralog_df1)

[1] "MT-ND2_MT-ND4" "MT-ND2_MT-ND5" "MT-ND4_MT-ND5" "MT-ND4_MT-ND2"
[5] "MT-ND5_MT-ND4" "MT-ND5_MT-ND2"

In [28]:
# get marker lists
sp1 = 'tree_shrew'
sp1_ctype = 'subclass'
sp2 = 'marmoset'
currstudy = 'sestan'

if(sp1 == 'tree_shrew' & sp1_ctype == 'subclass'){
    tab2 = fread('tree_shrew_ortho_subclass_markers.csv.gz')
    tab2 <- tab2[which(tab2$cell_type!='other'),]
    
}else if(sp1 == 'tree_shrew' & sp1_ctype != 'subclass'){
    tab2 = fread('tree_shrew_de_novo_markers.csv.gz')
    tab2 <- tab2[which(tab2$cell_type!='16'),]   # remove other cluster
    
}else{
    tab2 = fread(paste0(sp1, '_subclass_metaMarkers.csv.gz'))
}
ctypes2 = unique(tab2$cell_type)
tab2[1:2,]

tab3 = fread(paste0(currstudy, '_', sp2, '_de_novo_cluster_markers.csv.gz'))
ctypes3 = unique(tab3$cell_type)
tab3[1:2,]

group,cell_type,gene,fold_change,auroc,log_fdr,population_size,population_fraction,average_expression,se_expression,detection_rate,fold_change_detection,precision,recall
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
all,Chandelier,RORA,4.402665,0.9443124,-236.7660,213,0.04971989,7009.239,252.4487,0.9906103,1.179995,0.05815877,0.9906103
all,Chandelier,INPP4B,3.347238,0.9112779,-202.6675,213,0.04971989,6271.878,185.4359,0.9906103,1.231094,0.06052783,0.9906103


group,cell_type,gene,fold_change,auroc,log_fdr,population_size,population_fraction,average_expression,se_expression,detection_rate,fold_change_detection,precision,recall
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
all,0,KAZN,2.319694,0.9005982,-2185.326,2495,0.08739667,5289.255,33.78873,0.9991984,1.023573,0.08927484,0.9991984
all,0,FGF12,1.934287,0.8758221,-1923.478,2495,0.08739667,3393.967,22.20508,1.0000000,1.008625,0.08808473,1.0000000


In [29]:
# make 2 sets, one for each direction
# human - NHP
paralog_df1 = data.frame(genes = paste0(unlist(tab1[,1]), '_', unlist(tab1[,2])),
                        LCA = tab1[,3], homology_type = tab1[,4])
head(paralog_df1, n = 3)

# NHP - human
paralog_df2 = paralog_df1

,genes,LCA,homology_type
,<chr>,<chr>,<chr>
1,MT-ND2_MT-ND4,Bilateria,other_paralog
2,MT-ND2_MT-ND5,Bilateria,other_paralog
3,MT-ND4_MT-ND5,Bilateria,other_paralog


In [30]:
# get top 100 markers
tab2$rank = rep(1:(dim(tab2)[1]/length(ctypes2)), length(ctypes2))
top_markers1 = tab2[(tab2$rank<=100),]
tab3$rank = rep(1:(dim(tab3)[1]/length(ctypes3)), length(ctypes3))
top_markers2 = tab3[(tab3$rank<=100),]

In [31]:
combos = rbind(rep(ctypes2, each = length(ctypes3)), rep(ctypes3, length(ctypes2)))
dim(combos)
combos[1:2,1:4]

[1]   2 234

Chandelier,Chandelier,Chandelier,Chandelier
0,1,10,11


In [32]:
# get details of lca, spec scores
get_paralog_details <- function(vec1, vec2, pmat){
    mat1 = data.frame(g1 = rep(vec1, each = length(vec2)), 
                      g2 = rep(vec2, length(vec1)), homolog = NA, LCA = NA, homology_type = NA)
    glist = paste0(mat1[,1], '_', mat1[,2])
    temp2 = match(glist, pmat[,1])
    
    mat1$homolog = !is.na(temp2)
    mat1$LCA = pmat[temp2,2]
    mat1$homology_type = pmat[temp2,3]
    return(mat1)
}

In [33]:
newdf2 = c()
pb = txtProgressBar(min = 0, max = dim(combos)[2], initial = 0)

for(ii in 1:dim(combos)[2]){
    ctype1 = combos[1,ii]
    ctype2 = combos[2,ii]
    m1 = top_markers1$gene[top_markers1$cell_type==ctype1]
    m2 = top_markers2$gene[top_markers2$cell_type==ctype2]
    
    temp = intersect(m1, m2)
    
    if(length(temp)){
        rem_genes1 = setdiff(m1, temp)
        rem_genes2 = setdiff(m2, temp)

        # get orthologs
        list_ortholog = data.frame(g1 = temp, g2 = temp,
                                   homolog = TRUE, LCA = NA,
                                   homology_type = 'ortholog_one2one')   
        list_ortholog$cluster1 = ctype1
        list_ortholog$cluster2 = ctype2
        newdf2 = rbind(newdf2, list_ortholog)
    }else{
        rem_genes1 = m1
        rem_genes2 = m2
    }
    
    if(length(rem_genes1)){
        list1 = get_paralog_details(rem_genes1, m2, paralog_df1)
        list1$cluster1 = ctype1
        list1$cluster2 = ctype2
        newdf2 = rbind(newdf2, list1)
    }
    if(length(rem_genes2)){
        list2 = get_paralog_details(rem_genes2, m1, paralog_df2) 
        temp_list2 <- list2[,c(2,1,3:5)]
        colnames(temp_list2) = colnames(list2)
        list2 <- temp_list2
        
        list2$cluster1 = ctype1
        list2$cluster2 = ctype2        
        newdf2 = rbind(newdf2, list2)
    }
             
    setTxtProgressBar(pb, ii)
}

newdf2 <- newdf2[newdf2$homolog==TRUE,]
newdf2$homology[newdf2$homology_type!='ortholog_one2one'] = 'ortholog_many2many'

# removing LCA column since comparing among human paralogs
newdf2 <- newdf2[,-match(c('LCA', 'homology_type'), colnames(newdf2))]

newdf2 <- newdf2[!(duplicated(newdf2)),]
newdf2$species1 = sp1
newdf2$species2 = sp2
newdf2$primate_study = currstudy

dim(newdf2)
newdf2[1:5,]

# save
path_to_save = paste0('/vault/suresh/consensus_interneuron_taxonomy/', currstudy, '/')

if(sp1 == 'tree_shrew' & sp1_ctype=='subclass'){
    savefile = paste0(path_to_save, sp1, '_subclass_', sp2, '_', currstudy, '_marker_paralogs_list.csv')
}else if(sp1 == 'tree_shrew' & sp1_ctype!='subclass'){
    savefile = paste0(path_to_save, sp1, '_cluster_', sp2, '_', currstudy, '_marker_paralogs_list.csv')
}else{
    savefile = paste0(path_to_save, sp1, '_', sp2, '_', currstudy, '_marker_paralogs_list.csv')
}
    
write.table(newdf2, file = savefile, sep = ',', row.names = F, col.names = T, quote = F)

[1] 11980     9

,g1,g2,homolog,cluster1,cluster2,homology,species1,species2,primate_study
,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,CALN1,CALN1,TRUE,Chandelier,0,ortholog_one2one,tree_shrew,marmoset,sestan
2,ERBB4,ERBB4,TRUE,Chandelier,0,ortholog_one2one,tree_shrew,marmoset,sestan
3,MEF2C,MEF2C,TRUE,Chandelier,0,ortholog_one2one,tree_shrew,marmoset,sestan
4,PRKG1,PRKG1,TRUE,Chandelier,0,ortholog_one2one,tree_shrew,marmoset,sestan
5,SCN9A,SCN9A,TRUE,Chandelier,0,ortholog_one2one,tree_shrew,marmoset,sestan


In [134]:
currgene = 'GRM8'
match(currgene, tab2$gene[tab2$cell_type=='Sncg'])
match(currgene, tab2$gene[tab2$cell_type=='Vip'])
match(currgene, tab3$gene[tab3$cell_type=='20'])

[1] 56

[1] 164

[1] 44

In [135]:
# newdf2 = read.delim('mouse_human_marker_paralogs_list.csv', sep = ',')
newdf2[newdf2$cluster1=='Sncg' & newdf2$cluster2=='20',]

,g1,g2,homolog,cluster1,cluster2,homology,species1,species2,primate_study
,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2764389,CDH18,CDH18,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764390,CSMD1,CSMD1,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764391,NRXN1,NRXN1,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764392,LUZP2,LUZP2,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764393,THSD7A,THSD7A,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764394,DSCAM,DSCAM,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764395,GALNT13,GALNT13,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764396,NCAM2,NCAM2,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan
2764397,SNTG1,SNTG1,TRUE,Sncg,20,ortholog_one2one,tree_shrew,macaque,caglayan


In [10]:
# mapping mouse subclass vs sestan cluster
mapdf = read.delim('mouse_primate_Caglayan_mapping.csv', sep = ',')
tail(mapdf)

,cluster,subclass,class
,<int>,<chr>,<chr>
21,20,Sncg,CGE
22,8,Chandelier,MGE
23,11,Lamp5 Lhx6,CGE
24,25,Sst Chodl,MGE
25,1,Lamp5,CGE
26,15,Lamp5,CGE


In [11]:
# distance betn clusters by nbd
newdf2$nbd1 = mapdf$class[match(newdf2$cluster1, mapdf$subclass)]
newdf2$nbd2 = mapdf$class[match(newdf2$cluster2, mapdf$cluster)]

newdf2$sub1 = newdf2$cluster1
newdf2$sub2 = mapdf$subclass[match(newdf2$cluster2, mapdf$cluster)]

newdf2$dist = NA
newdf2$dist[newdf2$nbd1 != newdf2$nbd2] = 2
newdf2$dist[newdf2$nbd1 == newdf2$nbd2] = 1
newdf2$dist[newdf2$sub1 == newdf2$sub2] = 0

newdf2[1:2,]

,g1,g2,paralog,LCA,homology_type,cluster1,cluster2,nbd1,nbd2,sub1,sub2,dist
,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,PRKG1,PRKG1,TRUE,Eutheria,ortholog_one2one,Chandelier,0,MGE,MGE,Chandelier,Pvalb,1
2,TMEM108,TMEM108,TRUE,Eutheria,ortholog_one2one,Chandelier,0,MGE,MGE,Chandelier,Pvalb,1


In [12]:
table(tab1[,3])
table(newdf2$LCA)


                                Amniota                               Bilateria 
                                  19283                                  113006 
                          Boreoeutheria                              Catarrhini 
                                   2101                                    1622 
                               Chordata                        Euarchontoglires 
                                 275423                                   16436 
                           Euteleostomi                                Eutheria 
                                  88784                                   31641 
                                 Glires                           Gnathostomata 
                                     98                                   83249 
                            Haplorrhini                               Hominidae 
                                     48                                     281 
                           


         Amniota        Bilateria    Boreoeutheria         Chordata 
               8             3843               98              182 
Euarchontoglires     Euteleostomi         Eutheria    Gnathostomata 
             680               11              147               43 
    Opisthokonta       Vertebrata 
             511              219 

In [17]:
agegrp = 'Bilateria'
phyper(sum(newdf2$LCA==agegrp), sum(tab1[,3]==agegrp)/2, 
       (dim(tab1)[1] - sum(tab1[,3]==agegrp))/2, dim(newdf2)[1], lower.tail = F)

[1] 0